In [1]:
import pandas as pd
import csv
import os
import numpy as np
import boto3

In [2]:
# Importar dataset
df = pd.read_csv('train.csv', encoding='utf-8', engine='python', quoting=csv.QUOTE_MINIMAL, quotechar='"')

# Verificar se as linhas que no csv pareciam corrompidas ficaram bem
print(df.loc[9, "Review"])
print(df.loc[19, "Review"])

 Ok, we have all seen the negative reviews. Actually, one negative review copied by a lot of others...that obviously, never drove the car!!This car is one of the GREAT ONES! It has every creature comfort, plenty of power,a great roadster look, ease of entry, great roominess in the comfortable back seats, and has led to one GREAT SUMMER OF FUN.At this time, you won't see many on the road because that one reviewer scared the masses, but he will be made to look as stupid as those that dissed on the MUSTANG and the CORVETTE when they first came public. We know how that turned out!Kudos to ...and anyone that thinks this looks like a jeep..get GLASSES. ;))Five Stars from CT!!!
 Bought in 2012.Since then nothing but issues from the first week.At the outset there were several times it would not start.Window motors and regulators have been replaced; ignition starter; computer; electrical harness; passenger seat more than once; convertible top - nothing but problems - it is in again right now ne

In [3]:
print(df.loc[154, "Review"])
print(df.loc[155, "Review"])
print(df.loc[156, "Review"])
#print(df.loc[157, "Review"])

 My 2010 Nissan Rouge is one of the many 2008-2010 Nissan Rogue vehicles that are defected.  It has same continuous transmission problem, it is unsafe to drive in a highway.  However, Nissan does not do a recall.
Nissan will not help and cost is $4k plus wracked up $1k rental feeship and 6k miles with the mileage at 400 miles over the 120k warranty. Driving interstate, whining noise, rpms at red line with acceleration and then unable to maintain speed until complete stall. Restarted after cool down, check engine light came on 10 miles later but no problems while driving to dealer.
 This SUV has been a great combination of a family car and has been fun to drive.  We have had no mechanical issues in the past 5 years (except one tire pressure sensor became faulty and was repaired by the dealership free under warranty). Battery life, breaks, tires have all lasted 5 years. Currently has 45,000 miles. Accelerates well for an SUV, gets reasonable gas mileage, and looks sporty.


In [4]:
print(df.loc[235, "Review"])

 Reminds me of my old Kia, actually might be worse!s approaching 40,000 miles!elf after 3 hours in temps > 90 degrees!


In [5]:
def fix_multiline_csv(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as fin, open(output_path, 'w', encoding='utf-8') as fout:
        buffer = ""
        inside_quotes = False

        for line in fin:
            buffer += line.rstrip('\n')
            # Conta aspas não escapadas na linha
            quotes_count = buffer.count('"') - buffer.count('\\"')

            if quotes_count % 2 == 0:
                # Número par de aspas -> linha completa
                fout.write(buffer + '\n')
                buffer = ""
            else:
                # Aspas abertas, junta próxima linha
                buffer += ' '

fix_multiline_csv('train.csv', 'train_fixed.csv')

In [6]:
df = pd.read_csv(
    'train_fixed.csv',
    encoding='utf-8',
    engine='python',
    quoting=csv.QUOTE_MINIMAL,
    quotechar='"'
)

In [7]:
print(df.loc[154, "Review"])
print(df.loc[155, "Review"])
print(df.loc[156, "Review"])
#print(df.loc[157, "Review"])

 My 2010 Nissan Rouge is one of the many 2008-2010 Nissan Rogue vehicles that are defected.  It has same continuous transmission problem, it is unsafe to drive in a highway.  However, Nissan does not do a recall.
 I purchased my 2010 Nissan Rogue 6 months ago for $12k with 114k actual miles. Seemed like a great car, but after a month I noticed a shuddering at stop lights almost like it would stall. I had it checked at 3K mile oil change and tech said it was fine. 3 weeks ago the transmission failed completely after 6 months ownership and 6k miles with the mileage at 400 miles over the 120k warranty. Driving interstate, whining noise, rpms at red line with acceleration and then unable to maintain speed until complete stall. Restarted after cool down, check engine light came on 10 miles later but no problems while driving to dealer. Nissan will not help and cost is $4k plus wracked up $1k rental fee 
 This SUV has been a great combination of a family car and has been fun to drive.  We ha

In [8]:
print(df.loc[235, "Review"])

 Leased in Jan 2014  Problems:  Day 2: headlights needed to be adjusted, blinding on coming drivers.  Week 2: back windshield replaced; major leak  Month 2: GPS/radio system changed  Month 6: Air bag light went on for the day; eventually fixed itself  Month 6: All air system blocked; fan working but NO air coming out; fixed itself after 3 hours in temps > 90 degrees!  Month 12: Shifter assembly broken  Month 15: seat assembly broken (drivers side)  Month 16: CVT transmission grinding; replaced whole transmission  Now: have no choice but to get extended warranty as approaching 40,000 miles!  Reminds me of my old Kia, actually might be worse!


In [9]:
print(df.loc[11824, "Review"])

 This is my 6th Nissan (third Maxima) and my last. There is a constant rumbling vibration felt through the gas pedal and steering wheel between 20-45 mph. Dealer tried couldn't fix it, so rep came and said "this is normal for this transmission". NORMAL!? For a $30,000 car! They tout this new transmission as seamless, which apparently it's not. So, I'm now stuck with it! It's a design flaw Nissan won't admit to.


In [10]:
def fix_reviews_and_ratings(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as fin, \
         open(output_path, 'w', encoding='utf-8', newline='') as fout:

        reader = csv.reader(fin)
        writer = csv.writer(fout)

        header = next(reader)
        writer.writerow(header)

        review_idx = header.index("Review")
        rating_idx = header.index("Rating")
        num_cols = len(header)

        buffer_row = None

        for row in reader:
            # Garante o mesmo número de colunas que o cabeçalho
            while len(row) < num_cols:
                row.append("")

            if row[review_idx].strip():
                # Nova review começa aqui
                if buffer_row:
                    writer.writerow(buffer_row)
                buffer_row = row[:]
            else:
                # Linha de continuação da review anterior
                if not buffer_row:
                    continue

                extra_text = row[0].strip()
                if extra_text and not extra_text.isdigit():
                    buffer_row[review_idx] = (buffer_row[review_idx].strip() + " " + extra_text).strip()

                # Verifica se row[1] é um rating válido (segunda coluna!)
                second_field = row[1].strip()
                try:
                    float(second_field.replace(",", ""))
                    buffer_row[rating_idx] = second_field
                except ValueError:
                    pass  # Ignora se não for número

        # Escreve a última review se ainda houver uma no buffer
        if buffer_row:
            writer.writerow(buffer_row)

# Uso:
fix_reviews_and_ratings(
    'train_fixed.csv',
    'train_fixed_final.csv'
)

In [11]:
# Agora com o dataset sem erros podemos lê-lo novamente

df = pd.read_csv(
    'train_fixed_final.csv',
    encoding='utf-8',
    engine='python',
    quoting=csv.QUOTE_MINIMAL,
    quotechar='"'
)

In [12]:
df["Review"].duplicated().sum()

29

In [13]:
df = df.drop_duplicates(subset=["Review"], keep="first").reset_index(drop=True)
print("Reviews duplicadas restantes:", df["Review"].duplicated().sum())

Reviews duplicadas restantes: 0


In [14]:
# Dropar a primeira coluna/índice
df = df.drop(columns=['Unnamed: 0'])

# Verificar valores nulos
df.isnull().sum()

Review_Date      0
Author_Name      0
Vehicle_Title    0
Review_Title     2
Review           0
Rating           0
dtype: int64

In [15]:
df.dropna(inplace=True)

# Acrescentar coluna Real_Label com base no rating
def map_real_label(score):
    if score < 2.455:
        return "Negative"
    elif score <= 3.444:
        return "Neutral"
    else:
        return "Positive"

df["Real_Label"] = df["Rating"].apply(map_real_label)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11756 entries, 0 to 11757
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Review_Date    11756 non-null  object 
 1   Author_Name    11756 non-null  object 
 2   Vehicle_Title  11756 non-null  object 
 3   Review_Title   11756 non-null  object 
 4   Review         11756 non-null  object 
 5   Rating         11756 non-null  float64
 6   Real_Label     11756 non-null  object 
dtypes: float64(1), object(6)
memory usage: 734.8+ KB


In [17]:
# Guardar o ficheiro em csv (e alterar-lhe o nome) e json
df.to_csv("train_fixed_final.csv", index=False)
os.rename("train_fixed_final.csv", "Edmunds_Car_Ratings_final.csv")
df.to_json("Edmunds_Car_Ratings_final.json", orient="records", indent=4)

In [18]:
# Enviar ficheiros para o S3
s3_client = boto3.client('s3', region_name='eu-west-1')
bucket_name = 'i32419'

def upload_file(local_file_path, s3_path):
    s3_client.upload_file(local_file_path, bucket_name, s3_path)
    print(f"Arquivo {local_file_path} enviado para s3://{bucket_name}/{s3_path}")

upload_file('Edmunds_Car_Ratings_final.csv', 'datasets/Edmunds_Car_Ratings_final.csv')
upload_file('Edmunds_Car_Ratings_final.json', 'datasets/Edmunds_Car_Ratings_final.json')

Arquivo Edmunds_Car_Ratings_final.csv enviado para s3://i32419/datasets/Edmunds_Car_Ratings_final.csv
Arquivo Edmunds_Car_Ratings_final.json enviado para s3://i32419/datasets/Edmunds_Car_Ratings_final.json
